In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/content/GNSS_HighRain_Dataset.csv")

In [3]:
print(df.shape)
print(df.head())
print(df.info())
print(df.describe())

(2000, 20)
   Time(ms)    Temp(C)  Humidity(%)  RainAnalog  RainDigital  SNR1  SNR2  \
0     83400  25.468549    83.637278         884            1    33    30   
1    190800  25.449838    81.171019         861            1    31    29   
2    131600  23.869559    99.514493         807            1    27    32   
3    122400  23.287945    87.277936         734            1    39    37   
4    149400  26.386616    96.599739         777            1    32    40   

   SNR3  SNR4  SNR5  SNR6  SNR7  SNR8  SNR9  SNR10  SNR11  SNR12  SNR13  \
0    31    30    30     0     0     0     0      0      0      0      0   
1    30    32    33     0     0     0     0      0      0      0      0   
2    36    39    28     0     0     0     0      0      0      0      0   
3    34    32    33     0     0     0     0      0      0      0      0   
4    30    33    26     0     0     0     0      0      0      0      0   

   SNR14  SNR15  
0      0      0  
1      0      0  
2      0      0  
3      0 

In [4]:
df.isnull().sum()
(df == 0).sum()

,0
Time(ms),1
Temp(C),0
Humidity(%),0
RainAnalog,0
RainDigital,0
SNR1,0
SNR2,0
SNR3,0
SNR4,0
SNR5,0


In [5]:
snr_cols = [col for col in df.columns if 'SNR' in col]

for col in snr_cols:
    df[col] = df[col].apply(lambda x: np.nan if x > 100 else x)

In [6]:
for col in snr_cols:
    df[col] = df[col].replace(0, np.nan)

In [7]:
df[snr_cols] = df[snr_cols].interpolate(method='linear')

In [8]:
df[snr_cols] = df[snr_cols].fillna(method='bfill')
df[snr_cols] = df[snr_cols].fillna(method='ffill')

/tmp/ipython-input-3485316465.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[snr_cols] = df[snr_cols].fillna(method='bfill')
/tmp/ipython-input-3485316465.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[snr_cols] = df[snr_cols].fillna(method='ffill')


In [9]:
for col in snr_cols:
    df[col] = df[col].rolling(window=5, min_periods=1).mean()

In [10]:
df['Mean_SNR'] = df[snr_cols].mean(axis=1)

In [11]:
df['SNR_Var'] = df[snr_cols].var(axis=1)

In [12]:
df['Sat_Count'] = df[snr_cols].notnull().sum(axis=1)

In [13]:
df[['Temp(C)', 'Humidity(%)']] = df[['Temp(C)', 'Humidity(%)']].interpolate()

In [14]:
for col in snr_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.clip(df[col], lower, upper)

In [15]:
from sklearn.preprocessing import RobustScaler

features = df.drop(['RainDigital'], axis=1)

scaler = RobustScaler()
scaled = scaler.fit_transform(features)

X = pd.DataFrame(scaled, columns=features.columns)
y = df['RainDigital']

/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1424: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


In [16]:
snr_cols = [col for col in df.columns if "SNR" in col]

for col in snr_cols:
    df.loc[df[col] > 100, col] = np.nan

In [17]:
for col in snr_cols:
    df.loc[df[col] == 0, col] = np.nan

In [18]:
df[['Temp(C)', 'Humidity(%)']] = df[['Temp(C)', 'Humidity(%)']].interpolate()

In [19]:
for col in snr_cols:
    df[col] = df[col].rolling(window=5, min_periods=1).mean()

In [20]:
for col in snr_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

In [21]:
df['Mean_SNR'] = df[snr_cols].mean(axis=1)
df['SNR_Var'] = df[snr_cols].var(axis=1)
df['Sat_Count'] = df[snr_cols].notna().sum(axis=1)

In [22]:
snr_cols = [col for col in df.columns if "SNR" in col]

In [23]:
df[snr_cols] = df[snr_cols].round()

In [24]:
df[snr_cols] = df[snr_cols].astype("Int64")

In [25]:
print(df[snr_cols].head())
print(df[snr_cols].dtypes)

   SNR1  SNR2  SNR3  SNR4  SNR5  SNR6  SNR7  SNR8  SNR9  SNR10  SNR11  SNR12  \
0    33    30    31    30    30    28    31    38    37   <NA>   <NA>   <NA>   
1    32    30    31    30    31    28    31    38    37   <NA>   <NA>   <NA>   
2    32    30    31    32    31    28    31    38    37   <NA>   <NA>   <NA>   
3    32    30    32    32    31    28    31    38    37   <NA>   <NA>   <NA>   
4    32    31    32    32    31    28    31    38    37   <NA>   <NA>   <NA>   

   SNR13  SNR14  SNR15  Mean_SNR  SNR_Var  
0   <NA>   <NA>   <NA>        30       43  
1   <NA>   <NA>   <NA>        30       43  
2   <NA>   <NA>   <NA>        30       43  
3   <NA>   <NA>   <NA>        31       43  
4   <NA>   <NA>   <NA>        31       43  
SNR1        Int64
SNR2        Int64
SNR3        Int64
SNR4        Int64
SNR5        Int64
SNR6        Int64
SNR7        Int64
SNR8        Int64
SNR9        Int64
SNR10       Int64
SNR11       Int64
SNR12       Int64
SNR13       Int64
SNR14       Int64
SNR

In [26]:
df.to_csv("Rainy_Havvy_Clean.csv", index=False)